# Three protective-stop overlays

Read-only inspection of the completed release. No model fits, construction or missing-release rebuild. ATR, adjusted overnight gap and hybrid scales use a prior-year calibration with quarterly freezes. See the research contract and report qualifications. Results are conditional on each model’s fixed observed-date cohort; different models can have different dates.

**Current performance standard: 10 bp per executed entry/exit side (about 20 bp round trip at 100% gross).** Stops retain their original 1bp/side calibration. The new report reprices the saved fills; it does not recalibrate policies.

In [ ]:
from pathlib import Path
import json
import pandas as pd
from IPython.display import display, Image
PROJECT_DIR = Path.cwd()
if not (PROJECT_DIR / "data").exists():
    PROJECT_DIR = PROJECT_DIR.parent
RELEASE = PROJECT_DIR / "data/backtests/13_stop_overlays_v1"
if not (RELEASE / "MANIFEST.json").exists():
    raise FileNotFoundError("Completed stop release required; inspection never starts analysis or fits.")
manifest = json.loads((RELEASE / "MANIFEST.json").read_text())
assert manifest["complete"] and manifest["model_fits"] == 0
COST_REPORT = PROJECT_DIR / "data/reports/13_stop_overlays_10bps_v1"
if not (COST_REPORT / "MANIFEST.json").exists():
    raise FileNotFoundError("Completed 10bp report required; inspection does not rebuild it.")
cost_manifest = json.loads((COST_REPORT / "MANIFEST.json").read_text())
assert cost_manifest["complete"] and cost_manifest["standard_cost_bps_per_side"] == 10
assert cost_manifest["stop_calibration_cost_bps_per_side"] == 1
display(pd.DataFrame(manifest["model_catalog"]))

In [ ]:
metrics = pd.read_csv(COST_REPORT / "metrics.csv")
comparison = metrics.loc[metrics.cost_bps.eq(10), ["model", "n_per_side", "approach", "sessions", "annual_compound", "sharpe", "max_drawdown", "stop_rate", "worst_day", "expected_shortfall_5pct", "mean_cost_bps", "mean_gross_bps", "mean_net_bps"]]
display(comparison)
display(Image(filename=str(COST_REPORT / "stop_equity_comparison.png")))
display(Image(filename=str(COST_REPORT / "all_models_stop_sharpe.png")))

In [ ]:
policies = pd.read_csv(RELEASE / "quarterly_stop_policies.csv")
display(policies.groupby(["model", "family", "policy_status"]).size().rename("quarters").to_frame())
display(policies.tail(12))
display(pd.read_parquet(RELEASE / "excluded_comparison_dates.parquet").groupby("model").size().rename("omitted_dates").to_frame())
# Shared dates permit comparisons across all nine full-history models.
display(pd.read_csv(COST_REPORT / "nine_models_shared_dates_metrics.csv"))

## Execution sensitivity

The first replay produces some distances below one basis point and many first-bar stops. These are material limitations of the optimistic open-entry/barrier-fill assumptions. No executable price-increment, queue or extra stop slippage model has been imposed. The diagnostics below expose this sensitivity; they do not certify fills.

In [ ]:
import numpy as np
rows = []
for model in ("ret_joint30", "ret_rich42", "ret_tuned42", "ranker_corrected42"):
    trades = pd.read_parquet(RELEASE / f"{model}_trades.parquet")
    trades = trades.loc[trades.approach.ne("no_stop")].copy()
    # Reconstruct entry from exit price and its raw entry-to-exit return.
    trades["distance_bps"] = 10000 * trades.distance_price * (1 + trades.raw_return) / trades.fill_price
    for (n, approach), group in trades.groupby(["n_per_side", "approach"]):
        stopped = group.loc[group.stopped]
        rows.append(dict(model=model, n_per_side=n, approach=approach,
            stop_rate=group.stopped.mean(), median_distance_bps=group.distance_bps.median(),
            p10_distance_bps=group.distance_bps.quantile(.1), minimum_distance_bps=group.distance_bps.min(),
            first_bar_fraction_of_stops=stopped.exit_slot.eq(0).mean(),
            winners_cut_fraction=(stopped.direction * stopped.original_raw_return > 0).mean()))
display(pd.DataFrame(rows))